<a href="https://colab.research.google.com/github/preciousiajilore/BDG/blob/main/CNN_Question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **CMPUT 466/566 - CNN**

---




### **RESOURCES**
This assignment requires some basic knowledge of Pytorch which can be found in the following links:


1.   [Tensors](https://pytorch.org/tutorials/beginner/basics/tensorqs_tutorial.html)
2.   [Build the Neural Network](https://pytorch.org/tutorials/beginner/basics/buildmodel_tutorial.html)
3. [Optimizing Model Parameters](https://pytorch.org/tutorials/beginner/basics/optimization_tutorial.html)
4. [torch.nn](https://pytorch.org/docs/stable/nn.html)
5. [ResNet](https://pytorch.org/hub/pytorch_vision_resnet/)




### **DATASET**
The dataset is from [Kaggle](https://www.kaggle.com/datasets/die9origephit/children-vs-adults-images) and has been modified for some questions. The shared ***'balanced_data'*** folder contains the balanced dataset and ***'imbalanced_data'*** contains its modified version for Part IV with class imbalance.

The dataset can be found [here](https://drive.google.com/drive/folders/1WMYnKezCICtLNwnp9-eeK0220X6Hjlfm?usp=sharing)

Load the data on Google colab from Google drive:
1. Click on the Google Drive link of the **datasets** folder (make sure you login with your ualberta.ca email address)
2. Click on the drop down next to the name of the folder and select **Add Shortcut to Drive**

<center><img src='https://drive.google.com/uc?id=155oG91lcrqIxLQ1K3XJUzBLhrcOMmkqn' width="300"
     height="400" ></center>

4. Come back to the **copy of this colab notebook** and mount the drive by running the cell below


In [ ]:
from google.colab import drive
#make sure you give the necessary authorization for colab to access your Google Drive
drive.mount('/content/drive')

5. Click Connect to **Google Drive**


<center><img src='https://drive.google.com/uc?id=1vVU4anE7Wo_BRBDrW_dZ8_ck-BbMmbLR' width="800"
     height="150" ></center>


6. **Choose your ualberta.ca account**

<center><img src='https://drive.google.com/uc?id=1GkRAHOxpgAe4_Dsb0MY1ZdcR5RfyWxne' width="300"
     height="400" align='middle' ></center>

7. **Grant permission**


<center><img src='https://drive.google.com/uc?id=17qYsveplhjnynP-tmPQnhZSEmv_x3oa7' width="400"
     height="700" align='middle' ></center>


8. If you want to access a folder called **'datasets'**, you can do this with:



```
dataset_dir = '/content/drive/MyDrive/datasets'
```





In [ ]:
'''
Follow the above steps and include the paths for training and test datasets
'''
main_path = <INCLUDE_TRAIN_PATH_HERE> #ENTER PATH HERE
test_path = <INCLUDE_TEST_PATH_HERE> #ENTER PATH HERE

### **Part I:  Activation functions for CNN [ 6 marks ]**

In [ ]:
# Making Sure Numpy works correctly with Imgaug:
!pip install "numpy<2"

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 40.9 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
opencv-contrib-python 4.12.0.88 requires numpy<2.3.0,>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
shap 0.50.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jaxlib 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
pytensor 2.35.1 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python 4.12.0.88 requires numpy<2.3.0,>=2; python_

In [ ]:
#Loading necessary libraries
import numpy as np
import pandas as pd
import skimage.io
from skimage import color
from skimage import io
import glob
import cv2
from scipy.ndimage.interpolation import map_coordinates
from scipy.ndimage.filters import gaussian_filter
import matplotlib.pyplot as plt
from torch.nn.modules.loss import BCEWithLogitsLoss
from torch.optim import lr_scheduler, Adam, SGD
import torch
import torchvision
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import torch.nn as nn
import torch.utils.data as data
import numpy as np
import os
import glob
import time
from sklearn.metrics import balanced_accuracy_score
from torch.autograd import Variable
from torch.nn import Linear, CrossEntropyLoss, Sequential, Conv2d, MaxPool2d, Module, Softmax, BatchNorm2d, Dropout
from torch.nn.modules.conv import ConvTranspose2d, Conv2d
from google.colab.patches import cv2_imshow
from sklearn.metrics import accuracy_score
from sklearn.metrics import classification_report
from tqdm import tqdm
from torchvision import models
'''
Import any necessary libraries here to keep your code organized
'''
#import <some_library>
#from <something> import <something>

In [ ]:
'''
DO NOT ALTER THE FOLLOWING CODE
'''
'''
Add the line below
'''
torch.cuda.empty_cache()
torch.manual_seed(0)
'''
Add the above line
'''
my_transforms = transforms.Compose([transforms.Resize((224,224)),   transforms.ToTensor(), transforms.Normalize((0.5, 0.5, 0.5), (0.5, 0.5, 0.5)),])
BATCH_SIZE = 16
IMAGE_SIZE = 32
NUM_CHANNELS = 3
n_epochs = 50 # the cnn will be trained for 50 epochs
dataset = datasets.ImageFolder(root=main_path, transform=my_transforms)
dataset_size = dataset.__len__() #compute the length of the training dataset
train_count = int(dataset_size * 0.8) #divide the training dataset to training and validation splits
val_count = dataset_size - train_count # keep the training proportion to 1 if no validation is required
train_dataset, valid_dataset = data.random_split(dataset, [train_count, val_count]) #perform a random split on the dataset based on the train and validation proportion
y_train_indices = train_dataset.indices
y_train = [dataset.targets[i] for i in y_train_indices] #assign the labels or target variables to y_train (classes)
test_data = datasets.ImageFolder(test_path, transform=my_transforms)
'''
Following train, validation and test dataloaders will also be used in Part III: Resnets
'''
'''
'''
train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=2, shuffle=True)
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, num_workers=2, )
test_dataloader = torch.utils.data.DataLoader(test_data, batch_size=BATCH_SIZE, )

device = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu') #check for gpu
print('Using ',device,'for model training') #print the device status

In [ ]:
#Visualize some training images
'''
DO NOT ALTER THE FOLLOWING CODE
'''
real_batch = next(iter(train_dataloader))
plt.figure(figsize=(8,8))
plt.axis("off")
plt.title("Training Images")
plt.imshow(np.transpose(torchvision.utils.make_grid(real_batch[0].to(device)[:64], padding=2, normalize=True).cpu(),(1,2,0)))
plt.show()
plt.imshow(real_batch[0][5].permute(1,2,0), vmin=0, vmax=255)
plt.show()
print(f"Label: {real_batch[1][5]}")

In [ ]:
'''
DO NOT ALTER THE FOLLOWING CODE
'''
def make_train_step(model, optimizer, loss_fn):
  '''
  INPUT: model, optimizer, loss function
  OUTPUT: train step
  '''
  def train_step(x,y):
    '''
    This function is used to train the model and update the model parameters. Do not change this function
    '''
    #make prediction
    yhat = model(x)
    #enter train mode
    model.train()
    #compute loss
    loss = loss_fn(yhat,y)
    loss.backward()
    optimizer.step()
    optimizer.zero_grad()
    return loss
  return train_step

In [ ]:
'''
DO NOT ALTER THE FOLLOWING CODE
'''
def train_model(model,n_epochs, loss_fn, train_step):
  '''
  This is the main function which is used to train the model, update weights, calculate loss and save the best model
  '''
  train_losses = []
  val_losses = []
  epoch_train_losses = []
  epoch_val_losses = []
  for epoch in range(n_epochs):
    epoch_loss = 0
    for i ,data in tqdm(enumerate(train_dataloader), total = len(train_dataloader)): #iterate ove batches
      x_batch , y_batch = data
      x_batch = x_batch.to('cuda') #move to gpu
      y_batch = y_batch.unsqueeze(1).float() #convert target to same nn output shape
      y_batch = y_batch.to('cuda') #move to gpu
      loss = train_step(x_batch, y_batch)
      epoch_loss += loss/len(train_dataloader)
      train_losses.append(loss.cpu().detach().numpy())
    epoch_train_losses.append(epoch_loss)
    print('\nEpoch : {}, train loss : {}'.format(epoch+1,epoch_loss))
    #validation does not require gradient
    with torch.no_grad():
      cum_loss = 0
      for x_batch, y_batch in valid_dataloader:
        x_batch = x_batch.to('cuda')
        y_batch = y_batch.unsqueeze(1).float() #convert target to same nn output shape
        y_batch = y_batch.to('cuda')
        model.eval()#model to eval mode
        yhat = model(x_batch)
        val_loss = loss_fn(yhat,y_batch)
        cum_loss += loss/len(valid_dataloader)
        val_losses.append(val_loss.item())
      epoch_val_losses.append(cum_loss)
      print('Epoch : {}, val loss : {}'.format(epoch+1,cum_loss))
      best_loss = min(epoch_val_losses)
      #save best model
      if cum_loss <= best_loss:
        best_model_wts = model.state_dict()
  #load best model
  model.load_state_dict(best_model_wts)
  return model, train_losses,val_losses

def plot_losses(train_losses,val_losses):
  '''
  This function can be used to plot the training and validation losses. You can use this
  function to analyse the losses and judge if model was overfitting or if model shows some
  unusual behaviour.
  '''
  plt.plot(train_losses, label='Training loss')
  plt.plot(val_losses, label='Validation loss')
  plt.legend()
  plt.show()

def inference(model,test_data):
  '''
  As we are doing binary classification, this function uses sigmoid to change class probabilities
  to either 0 or 1 class.
  '''
  y_pred = []
  y_true = []
  for idx in range(1, len(test_data)):
    y_true.append( test_data[idx][1])
    sample = torch.unsqueeze(test_data[idx][0], dim=0).to('cuda')
    if torch.sigmoid(model(sample)) < 0.5:
      y_pred.append(0)
    else:
      y_pred.append(1)
  return y_pred, y_true

def calc_loss(model, n_epochs):
  '''
  This function drives the training function, assigns the loss fuction and sets the optimiizer.
  '''
  loss_fn = BCEWithLogitsLoss()
  optimizer = torch.optim.Adam(model.parameters())
  train_step = make_train_step(model, optimizer, loss_fn)
  trained_model, train_losses, val_losses = train_model(model,n_epochs, loss_fn, train_step)
  return trained_model

def calc_accuracy(trained_model):
  '''
  This function is used for returning the calculated accuracies.
  '''
  y_pred, y_true = inference(trained_model,test_data)
  target_names = ['Adults', 'Kids']
  print('the accuracy is',accuracy_score(y_true, y_pred))
  print(classification_report(y_true, y_pred, target_names=target_names))
  print('the balanced accuracy is',balanced_accuracy_score(y_true, y_pred))
  return accuracy_score(y_true, y_pred)

#### **Consider the following code snippet for a Neural Network**

This Network is a very simple Network for your reference to implement a Neural Network of any given architecture.


```
class Net(Module):   
    
    def __init__(self):
        super(Net, self).__init__()
        self.cnn_layers = Sequential(
            # Defining a 2D convolution layer
            Conv2d(NUM_CHANNELS, IMAGE_SIZE, kernel_size=3, stride=1, padding=1),
            BatchNorm2d(IMAGE_SIZE),
            MaxPool2d(kernel_size=2, stride=2),
            # Defining another 2D convolution layer
            Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            BatchNorm2d(32),
            MaxPool2d(kernel_size=2, stride=2),
        )
        self.linear_layers = Sequential(
            Linear(100352, 1)
        )
    # Defining the forward pass    
    def forward(self, x):
        x = self.cnn_layers(x)
        x = x.view(x.size(0), -1)
        x = self.linear_layers(x)
        return x

# defining the model
model = Net()
# defining the optimizer
optimizer = Adam(model.parameters(), lr=0.07)
# defining the loss function
criterion = CrossEntropyLoss()
# checking if GPU is available
if torch.cuda.is_available():
    model = model.cuda()
    criterion = criterion.cuda()
    
print(model)

trained_model = calc_loss(model, n_epochs)
calc_accuracy(trained_model)
```



#### **(1) Build a CNN for the following Model Architecture [3 marks]**


```
Net(
  (cnn_1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (cnn_2): Sequential(
    (0): Conv2d(32, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (cnn_3): Sequential(
    (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_4): Sequential(
    (0): Conv2d(512, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_5): Sequential(
    (0): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_6): Sequential(
    (0): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fully_1): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=50176, out_features=4096, bias=True)
  )
  (fully_2): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=4096, out_features=4096, bias=True)
  )
  (fully_3): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=4096, out_features=1000, bias=True)
  )
  (fully_4): Sequential(
    (0): Linear(in_features=1000, out_features=1, bias=True)
  )
)
```

The codebase is as follows:


In [ ]:
from torch.nn.modules.conv import ConvTranspose2d, Conv2d
class Net(Module):
    def __init__(self):
        super(Net, self).__init__()
        '''
        Write the 6 CNNs and 4 fully connected CNNs here:
        '''
        #ENTER CODE HERE
    def forward(self, x):
        '''
        Define the forward pass
        '''
        #ENTER CODE HERE
        return x
'''
DO NOT ALTER THE FOLLOWING CODE
'''
model = Net()
optimizer = Adam(model.parameters(), lr=0.07)
criterion = CrossEntropyLoss()
if torch.cuda.is_available():
    model = model.cuda()
    criterion = criterion.cuda()
print(model)
trained_model = calc_loss(model, n_epochs) #train the model
calc_accuracy(trained_model) # report the accuracy

#### **(2) Activation Functions [3 marks]**
Plug in the following Activation Functions:

1. ReLU
2. SiLU
3. Sigmoid
4. Tanh
5. ELU


Your Network Architecture should be as follows:



```
Net(
  (cnn_1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (cnn_2): Sequential(
    (0): Conv2d(32, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (cnn_3): Sequential(
    (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_4): Sequential(
    (0): Conv2d(512, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_5): Sequential(
    (0): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_6): Sequential(
    (0): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fully_1): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=50176, out_features=4096, bias=True)
    (2): YOUR ACTIVATION FUNCTION COMES HERE
  )
  (fully_2): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=4096, out_features=4096, bias=True)
    (2): YOUR ACTIVATION FUNCTION COMES HERE
  )
  (fully_3): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=4096, out_features=1000, bias=True)
    (2): YOUR ACTIVATION FUNCTION COMES HERE
  )
  (fully_4): Sequential(
    (0): Linear(in_features=1000, out_features=1, bias=True)
  )
)
```








In [ ]:
'''
Build the Network Architecture: (keep in mind the activation functions)
Try it out for ReLU, SiLU, Sigmoid, Tanh, ELU individually
'''
from torch.nn.modules.conv import ConvTranspose2d, Conv2d
class Net(Module):
    def __init__(self):
        super(Net, self).__init__()
        '''
        Write the 6 CNNs and 4 fully connected CNNs here
        '''
        #ENTER CODE HERE
    def forward(self, x):
        '''
        Define the forward pass :
        '''
        #ENTER CODE HERE
        return x
'''
DO NOT ALTER THE FOLLOWING CODE
'''
model = Net()
optimizer = Adam(model.parameters(), lr=0.07)
criterion = CrossEntropyLoss()
if torch.cuda.is_available():
    model = model.cuda()
    criterion = criterion.cuda()
print(model)
trained_model = calc_loss(model, n_epochs) #train the model
calc_accuracy(trained_model) # report the accuracy

1. **Plot the accuracies for each activation function**

In [ ]:
#ENTER CODE HERE

2. **Which function performs better? Justify.**

ANSWER-

### **Part II: Custom Activation Functions**

#### **(1) Implement any activation function of your OWN and DO NOT USE any predefined PyTorch Activation Functions [ 3 marks]**



```
class custom_activation_function(nn.Module):
    '''
    Implementation of custom activation function
    '''
    def __init__(self, in_features, alpha = None):
        '''
        Initialization
        '''

    def forward(self, x):
        '''
        Forward pass of the function.
        '''

af = custom_activation_function(<some_parameters>)
x = torch.randn(256) # random tensor
x = af(x)
```

Your Network Architecture should be as follows:

```
Net(
  (cnn_1): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(32, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (cnn_2): Sequential(
    (0): Conv2d(32, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (cnn_3): Sequential(
    (0): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(512, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_4): Sequential(
    (0): Conv2d(512, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(256, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_5): Sequential(
    (0): Conv2d(256, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(128, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  )
  (cnn_6): Sequential(
    (0): Conv2d(128, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (fully_1): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=50176, out_features=4096, bias=True)
  )
  (fully_2): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=4096, out_features=4096, bias=True)
  )
  (fully_3): Sequential(
    (0): Dropout(p=0.5, inplace=False)
    (1): Linear(in_features=4096, out_features=1000, bias=True)
  )
  (fully_4): Sequential(
    (0): Linear(in_features=1000, out_features=1, bias=True)
  )
  (a1): custom_activation_function()
  (a2): custom_activation_function()
  (a3): custom_activation_function()
)
```

Hint: Here, we are asking you to apply Custom activation functions to the Fully Connected Layers in the forward pass of the Network


#### **(2) Implement any COMPLEX activation function of your OWN and DO NOT USE any predefined PyTorch Activation Functions {CMPUT 566 only} [ 5 marks]**

Implement any one of the following activation functions

1. [Soft exponential](https://arxiv.org/pdf/1602.01321.pdf)
2. [BReLU](https://arxiv.org/pdf/1709.04054.pdf)





### **Part III: ResNet [6 marks]**

#### **(1) Implement the following **pretrained ResNet variants****


1. ResNet18
2. ResNet50
3. ResNet152

You can refer the ResNet documentation in the RESOURCES tab.


In [ ]:
model_resnet_variant = <assign this to a pretrained resnet model> #ENTER CODE HERE

'''
DO NOT ALTER THE FOLLOWING CODE
'''
#freeze all the model parameters
for params in model_resnet_variant.parameters():
  params.requires_grad_ = False
'''
DO NOT ALTER THE ABOVE CODE
'''

nr_filters = <input features of last fully connected layer> #ENTER CODE HERE
model_resnet_variant.fc = nn.Linear(nr_filters, 1)
model_resnet_variant = model_resnet_variant.to(device)

trained_model = calc_loss(model_resnet_variant, n_epochs)
calc_accuracy(trained_model)

#### **(2) Which ResNet performs better? Justify.**

Your answer -  

### **Part IV: Class Imbalance and Sampling [ 5 marks]**




**Consider the imbalanced data and run the following CNN with and without Weighted Random Sampler**



```
class Net(Module):   
    def __init__(self):
        super(Net, self).__init__()

        self.cnn_layers = Sequential(
            # Defining a 2D convolution layer
            Conv2d(NUM_CHANNELS, IMAGE_SIZE, kernel_size=3, stride=1, padding=1),
            BatchNorm2d(IMAGE_SIZE),
            ReLU(inplace=True),
            MaxPool2d(kernel_size=2, stride=2),
            # Defining another 2D convolution layer
            Conv2d(32, 32, kernel_size=3, stride=1, padding=1),
            BatchNorm2d(32),
            ReLU(inplace=True),
            MaxPool2d(kernel_size=2, stride=2),
        )

        self.linear_layers = Sequential(
            Linear(100352, 1)
        )

    # Defining the forward pass    
    def forward(self, x):
        x = self.cnn_layers(x)
        x = x.view(x.size(0), -1)
        x = self.linear_layers(x)
        #x = x.view(x.size(0), -1)

        return x
        
```


**NOTE:**
1. Change the main and test paths to the imbalanced dataset.
2. Sampler can be loaded to data loader as follows:



```
sampler_W = <ENTER CODE for weighted sampler>

train_dataloader = DataLoader(train_dataset, batch_size=BATCH_SIZE, num_workers=2, sampler = sampler_W)
valid_dataloader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, num_workers=2)
```

**Has the accuracy gone up or down? Why? Explain your answer.**


### **Part V: Data Augmentation{CMPUT 466 only}[ 5 marks]**
1. **What is Data Augmentation? How does it help in combating the data imbalance issue?**

Answer -

2. **Rotate** the image clockwise by **30** degrees

In [ ]:
# importing an image from the imbalanced dataset
image_path = <import any random image from the dataset> #ENTER THE PATH
'''
DO NOT ALTER THE FOLLOWING CODE
HOWEVER, IF YOU WANT TO ANOTHER LIBRARY, FEEL FREE TO USE IT TOO.
JUST MAKE SURE TO DO THE AUGMENTATION AND THEN SHOW THE AUGMENTED IMAGE
'''

import imageio
import imgaug as ia
import imgaug.augmenters as iaa
%matplotlib inline
image = imageio.imread(image_path)
ia.imshow(image)

# Your code to rotate the image then show it. Use ia.imshow(image) if using imgaug library

##

3. Add **Gaussian noise** to the image.


In [ ]:
## YOUR CODE TO ADD GAUSSIAN NOISE TO THE IMAGE

4. Convolve the image by applying [**Sobel Edge Detection**](https://www.projectrhea.org/rhea/index.php/An_Implementation_of_Sobel_Edge_Detection) filter. You can apply either X or Y directional kernel and use any library.


In [ ]:
## YOUR CODE FOR SOBEL EDGE DETECTION

5. Think of **two cases** where augmentations might **not be useful**. For instance, in digits classification, applying a flipping augmentation for the digit 2 might be of no use.

Answer -

References

Credit:  Winter 2022 course materials.